In [2]:

# 1. IMPORT LIBRARIES

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# 2. DEVICE CONFIG

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# 3. DATASET PATH (KAGGLE)

DATA_DIR = "/kaggle/input/datasets/subhajournal/busi-breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"

# 4. IMAGE TRANSFORMS (with normalization)

# Normalization stats (ImageNet)
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

base_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

augment_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])


# 5. LOAD DATASET

dataset = datasets.ImageFolder(DATA_DIR, transform=base_transform)

indices = np.arange(len(dataset))
labels = np.array([label for _, label in dataset.samples])


# 6. TRAIN VALID TEST SPLIT (stratified)

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=labels[temp_idx],
    random_state=42
)

train_set = Subset(dataset, train_idx)
val_set   = Subset(dataset, val_idx)
test_set  = Subset(dataset, test_idx)

# We'll create dataloaders inside each experiment (they may differ)
num_classes = len(dataset.classes)

# 7. MODEL CREATION (without pretrained weights)

def create_network():
    net = models.resnet18(weights=None)
    in_features = net.fc.in_features
    net.fc = nn.Linear(in_features, num_classes)
    return net.to(DEVICE)


# 8. TRAINING FUNCTION (with validation and early stopping)

def train_model(model, train_loader, val_loader, loss_fn, optimizer, scheduler, epochs=30, patience=5):
    best_val_acc = 0.0
    best_epoch = 0
    patience_counter = 0

    for epoch in range(epochs):
        # Training
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_acc = 100. * correct / total
        train_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                outputs = model(images)
                loss = loss_fn(outputs, targets)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += targets.size(0)
                val_correct += predicted.eq(targets).sum().item()

        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / len(val_loader)

        # Learning rate scheduling
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        # Check if best validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model_temp.pth')
            patience_counter = 0
        else:
            patience_counter += 1

        # Print every 5 epochs
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    print(f"Best validation accuracy: {best_val_acc:.2f}% at epoch {best_epoch+1}")
    model.load_state_dict(torch.load('best_model_temp.pth'))
    return model


# 9. EVALUATION FUNCTION

def evaluate_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
    return y_true, y_pred


# 10. EXPERIMENT 1: BASELINE

print("\n" + "="*60)
print("EXPERIMENT 1: BASELINE")
print("="*60)

# Dataloaders
train_loader_base = DataLoader(Subset(dataset, train_idx), batch_size=32, shuffle=True)
val_loader_base   = DataLoader(Subset(dataset, val_idx),   batch_size=32, shuffle=False)
test_loader       = DataLoader(Subset(dataset, test_idx),  batch_size=32, shuffle=False)

model_base = create_network()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_base.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

model_base = train_model(model_base, train_loader_base, val_loader_base, loss_fn, optimizer, scheduler, epochs=30)

y_true, y_pred = evaluate_model(model_base, test_loader)
report_base = classification_report(y_true, y_pred, output_dict=True)
print("\nBaseline Classification Report")
print(classification_report(y_true, y_pred))


# 11. EXPERIMENT 2: CLASS WEIGHTING

print("\n" + "="*60)
print("EXPERIMENT 2: CLASS WEIGHTING")
print("="*60)

class_counts = np.bincount(labels[train_idx])
weights = 1.0 / class_counts
class_weights = torch.FloatTensor(weights).to(DEVICE)

weighted_loss = nn.CrossEntropyLoss(weight=class_weights)

model_weight = create_network()
optimizer = optim.Adam(model_weight.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

model_weight = train_model(model_weight, train_loader_base, val_loader_base, weighted_loss, optimizer, scheduler, epochs=30)

y_true, y_pred = evaluate_model(model_weight, test_loader)
report_weight = classification_report(y_true, y_pred, output_dict=True)
print("\nClass Weighted Report")
print(classification_report(y_true, y_pred))


# 12. EXPERIMENT 3: OVERSAMPLING (WeightedRandomSampler)

print("\n" + "="*60)
print("EXPERIMENT 3: OVERSAMPLING")
print("="*60)

sample_weights = weights[labels[train_idx]]
sampler = WeightedRandomSampler(
    torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader_over = DataLoader(Subset(dataset, train_idx), batch_size=32, sampler=sampler)

model_over = create_network()
optimizer = optim.Adam(model_over.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

model_over = train_model(model_over, train_loader_over, val_loader_base, loss_fn, optimizer, scheduler, epochs=30)

y_true, y_pred = evaluate_model(model_over, test_loader)
report_over = classification_report(y_true, y_pred, output_dict=True)
print("\nOversampling Report")
print(classification_report(y_true, y_pred))


# 13. EXPERIMENT 4: DATA AUGMENTATION

print("\n" + "="*60)
print("EXPERIMENT 4: DATA AUGMENTATION")
print("="*60)

# Create augmented training dataset
aug_dataset = datasets.ImageFolder(DATA_DIR, transform=augment_transform)
train_aug = Subset(aug_dataset, train_idx)
train_loader_aug = DataLoader(train_aug, batch_size=32, shuffle=True)

model_aug = create_network()
optimizer = optim.Adam(model_aug.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

model_aug = train_model(model_aug, train_loader_aug, val_loader_base, loss_fn, optimizer, scheduler, epochs=30)

y_true, y_pred = evaluate_model(model_aug, test_loader)
report_aug = classification_report(y_true, y_pred, output_dict=True)
print("\nAugmentation Report")
print(classification_report(y_true, y_pred))


# 14. EXPERIMENT 5: FOCAL LOSS

print("\n" + "="*60)
print("EXPERIMENT 5: FOCAL LOSS")
print("="*60)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * ((1 - pt) ** self.gamma) * ce_loss
        else:
            focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# Use class weights as alpha for focal loss
focal_loss_fn = FocalLoss(gamma=2, alpha=class_weights)

model_focal = create_network()
optimizer = optim.Adam(model_focal.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

model_focal = train_model(model_focal, train_loader_base, val_loader_base, focal_loss_fn, optimizer, scheduler, epochs=30)

y_true, y_pred = evaluate_model(model_focal, test_loader)
report_focal = classification_report(y_true, y_pred, output_dict=True)
print("\nFocal Loss Report")
print(classification_report(y_true, y_pred))


# 15. COMPARISON TABLE (F1 SCORES)

results = pd.DataFrame({
    "Method": ["Baseline", "Class Weighting", "Oversampling", "Augmentation", "Focal Loss"],
    "Macro F1": [
        report_base["macro avg"]["f1-score"],
        report_weight["macro avg"]["f1-score"],
        report_over["macro avg"]["f1-score"],
        report_aug["macro avg"]["f1-score"],
        report_focal["macro avg"]["f1-score"]
    ],
    "Weighted F1": [
        report_base["weighted avg"]["f1-score"],
        report_weight["weighted avg"]["f1-score"],
        report_over["weighted avg"]["f1-score"],
        report_aug["weighted avg"]["f1-score"],
        report_focal["weighted avg"]["f1-score"]
    ]
})

print("\n" + "="*60)
print("F1-SCORE COMPARISON")
print("="*60)
print(results.to_string(index=False))

Device: cuda

EXPERIMENT 1: BASELINE
Epoch  5 | Train Loss: 0.5574 | Train Acc: 76.54% | Val Loss: 0.7453 | Val Acc: 62.45%
Epoch 10 | Train Loss: 0.4099 | Train Acc: 82.52% | Val Loss: 0.5105 | Val Acc: 75.53%
Early stopping at epoch 12
Best validation accuracy: 79.75% at epoch 7

Baseline Classification Report
              precision    recall  f1-score   support

           0       0.83      0.81      0.82       134
           1       0.63      0.79      0.70        63
           2       0.77      0.50      0.61        40

    accuracy                           0.76       237
   macro avg       0.74      0.70      0.71       237
weighted avg       0.76      0.76      0.75       237


EXPERIMENT 2: CLASS WEIGHTING
Epoch  5 | Train Loss: 0.5937 | Train Acc: 71.56% | Val Loss: 1.0928 | Val Acc: 55.70%
Epoch 10 | Train Loss: 0.4148 | Train Acc: 80.80% | Val Loss: 1.4532 | Val Acc: 61.60%
Epoch 15 | Train Loss: 0.2821 | Train Acc: 86.87% | Val Loss: 0.5253 | Val Acc: 72.57%
Epoch 20 | Tr